## Homework 4


### Preparation

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
print(len(documents))

72


In [7]:
!echo $PREFIX

In [3]:
#PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"


SyntaxError: invalid syntax (1717744948.py, line 1)

In [8]:
#!wget ${PREFIX}../01-agentic-rag/code/rag_helper.py


In [9]:
#!wget ${PREFIX}/04-evaluation/code/evaluation_utils.py

In [10]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

## Q1. Generating questions
Generating questions for all 72 pages costs money and takes time, so let's start small and generate questions for just the first 3 pages:
  
01-agentic-rag/lessons/01-intro.md  
01-agentic-rag/lessons/02-environment.md  
01-agentic-rag/lessons/03-rag.md  

Each call returns the token usage, which most LLM APIs report on the response object (e.g. response.usage.input_tokens / prompt_tokens).  

What's the average number of input tokens across these 3 calls?  

140  
1400  
14000  
140000   

## Q2. First result with text search  


In [24]:
import pandas as pd
df=pd.read_csv('ground-truth.csv')

In [25]:
ground_truth = df.to_dict(orient="records")

We search over the same chunks as in homework 2.

Create them with chunk_documents:

In [26]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

This gives 295 chunks.  

Now rebuild the search from homework 2 over these chunks. Build a text index (Index) and a vector index (VectorSearch), both keyed on filename. Wrap each one in a function, text_search and vector_search, that takes a query and the number of results to return (5 by default).  
  
For hybrid search, reuse the rrf function from homework 2:  

In [27]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

Then define hybrid_search on top of it:

In [28]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [29]:
# Take the first question from the ground truth:
q = ground_truth[0]["question"]

### After running text_search for it, what's the filename of the first result?  

01-agentic-rag/lessons/01-intro.md  
01-agentic-rag/lessons/03-rag.md  
01-agentic-rag/lessons/13-function-calling.md  
01-agentic-rag/lessons/10-rag-next-steps.md  